# 00：建立 PERMNO 與 ticker 歷史名稱清單

來源：`Data/raw/stock/RAW_name.csv`。原始資料不修改。

## 輸出與明確假設
- `PERMNO_list.txt`：從 ticker 長度 >2 且日期與研究期間重疊的篩選結果取出 PERMNO，去重並按數值排序，一行一個，不加標題或逗號。
- `Ticker_name.txt`：Tab 分隔並附標題，欄位依序為 `ticker`、`secinfostartdt`、`secinfoenddt`、`permno`。
- ticker 去除前後空白後，長度必須大於 2。空 ticker 也會被排除；不額外要求純字母、不排除 GME、不另加證券種類限制。
- 研究期間為 **2012-01-01 至 2022-12-31，含兩端**。保留 `開始日 <= 2022-12-31 且 結束日 >= 2012-01-01` 的紀錄；只要有一天重疊就保留。
- 保留來源的完整起止時間，不裁切至研究期間；Ticker_name 的歷史紀錄不額外去重或合併，以免改變資料定義。
- PERMNO 必須為非空數字，日期必須有效且開始日不晚於結束日；遇到異常停止，不自動推測缺失日期。
- 分批讀取，輸出寫入 `Data/process/00_biild_ticker_list/<UTC時間>/`；執行摘要顯示筆數、日期範圍、unique PERMNO、unique ticker。此資料不是 Reddit 留言，unique authors 與 threads 為 N/A。

In [1]:
from pathlib import Path
from datetime import date, datetime, timezone
import csv, json, hashlib
import pandas as pd
from IPython.display import display

ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'Data/raw/stock/RAW_name.csv').is_file())
SOURCE=ROOT/'Data/raw/stock/RAW_name.csv'
START=date(2012,1,1)
END=date(2022,12,31)
CHUNK_SIZE=20_000
OUT=ROOT/'Data/process/00_biild_ticker_list'/datetime.now(timezone.utc).strftime('%Y-%m-%d_%H-%M-%S_UTC')
OUT.mkdir(parents=True,exist_ok=False)
input_stat=(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
print('來源：',SOURCE)
print('輸出：',OUT)

def overlaps(start,end):
    return start<=END and end>=START
assert overlaps(date(2011,1,1),START)
assert overlaps(END,date(2023,1,1))
assert not overlaps(date(2010,1,1),date(2011,12,31))
assert not overlaps(date(2023,1,1),date(2024,1,1))


來源： C:\Users\eagle\RedditStock\Data\raw\stock\RAW_name.csv
輸出： C:\Users\eagle\RedditStock\Data\process\00_biild_ticker_list\2026-10-05_08-12-26_UTC


## 分批篩選與輸出
先篩選 ticker 長度 >2 與日期重疊，兩份檔案均來自同一篩選結果。日期範圍保留來源完整起止時間。

In [2]:
def empty_stats():
    return {'rows':0,'start_min':None,'end_max':None,'permnos':set(),'tickers':set()}
stats={stage:empty_stats() for stage in ['input','ticker_length_gt_2','overlaps_2012_2022']}
def update(stage,ticker,permno,start,end):
    s=stats[stage];s['rows']+=1;s['permnos'].add(permno)
    if ticker:s['tickers'].add(ticker)
    s['start_min']=min(s['start_min'],start) if s['start_min'] else start
    s['end_max']=max(s['end_max'],end) if s['end_max'] else end
def diagnostic(stage):
    s=stats[stage]
    return {'stage':stage,'rows':s['rows'],'start_min':s['start_min'].isoformat() if s['start_min'] else None,
            'end_max':s['end_max'].isoformat() if s['end_max'] else None,
            'unique_permnos':len(s['permnos']),'unique_tickers':len(s['tickers']),
            'unique_authors':'N/A','unique_threads':'N/A'}

ticker_path=OUT/'Ticker_name.txt'
partial=ticker_path.with_suffix('.txt.partial')
fields=['ticker','secinfostartdt','secinfoenddt','permno']
with partial.open('w',encoding='utf-8',newline='') as out:
    writer=csv.writer(out,delimiter='\t');writer.writerow(fields)
    for batch_no,chunk in enumerate(pd.read_csv(SOURCE,usecols=fields,dtype=str,keep_default_na=False,
                        na_filter=False,encoding='utf-8-sig',chunksize=CHUNK_SIZE),1):
        for row in chunk[fields].itertuples(index=False,name=None):
            ticker,raw_start,raw_end,permno=row
            ticker=ticker.strip();permno=permno.strip()
            assert permno and permno.isascii() and permno.isdigit(), ('Invalid PERMNO',row)
            start=date.fromisoformat(raw_start);end=date.fromisoformat(raw_end)
            assert start<=end, ('Invalid date interval',row)
            update('input',ticker,permno,start,end)
            if len(ticker)<=2:continue
            update('ticker_length_gt_2',ticker,permno,start,end)
            if not overlaps(start,end):continue
            update('overlaps_2012_2022',ticker,permno,start,end)
            writer.writerow([ticker,raw_start,raw_end,permno])
        print('chunk',batch_no,'input rows',len(chunk))
        display(pd.DataFrame([diagnostic(stage) for stage in stats]))
partial.replace(ticker_path)
permnos=sorted(stats['overlaps_2012_2022']['permnos'],key=int)
permno_path=OUT/'PERMNO_list.txt'
permno_path.write_text(''.join(p+'\n' for p in permnos),encoding='utf-8')


chunk 1 input rows 20000


,stage,rows,start_min,end_max,unique_permnos,unique_tickers,unique_authors,unique_threads
0,input,20000,1925-12-31,2025-12-31,4622,5293,N/A,N/A
1,ticker_length_gt_2,18120,1962-07-02,2025-12-31,4094,5071,N/A,N/A
2,overlaps_2012_2022,5280,2008-06-27,2023-09-28,1727,1981,N/A,N/A


chunk 2 input rows 20000


,stage,rows,start_min,end_max,unique_permnos,unique_tickers,unique_authors,unique_threads
0,input,40000,1925-12-31,2025-12-31,9566,10430,N/A,N/A
1,ticker_length_gt_2,36047,1945-12-31,2025-12-31,8670,10069,N/A,N/A
2,overlaps_2012_2022,9486,2008-06-27,2023-09-28,3592,4007,N/A,N/A


chunk 3 input rows 20000


...

chunk 5 input rows 20000


,stage,rows,start_min,end_max,unique_permnos,unique_tickers,unique_authors,unique_threads
0,input,100000,1925-12-31,2025-12-31,24796,24729,N/A,N/A
1,ticker_length_gt_2,93894,1945-12-31,2025-12-31,23273,24216,N/A,N/A
2,overlaps_2012_2022,14178,2008-06-27,2023-09-28,5628,6193,N/A,N/A


chunk 6 input rows 10365


,stage,rows,start_min,end_max,unique_permnos,unique_tickers,unique_authors,unique_threads
0,input,110365,1925-12-31,2025-12-31,26907,26216,N/A,N/A
1,ticker_length_gt_2,104005,1945-12-31,2025-12-31,25337,25694,N/A,N/A
2,overlaps_2012_2022,16508,2008-06-27,2023-09-28,6675,7314,N/A,N/A


40050

## 重新讀取輸出並驗證
檢查 PERMNO 無重複、每行一值，以及所有 ticker 歷史紀錄均符合長度、日期重疊與來源 PERMNO 條件。

In [3]:
read_permnos=permno_path.read_text(encoding='utf-8').splitlines()
assert read_permnos==permnos and len(read_permnos)==len(set(read_permnos))
assert all(',' not in p and p.isdigit() for p in read_permnos)
checked=0
output_permnos=set()
preview=[]
with ticker_path.open(encoding='utf-8',newline='') as f:
    reader=csv.DictReader(f,delimiter='\t')
    assert reader.fieldnames==fields
    for row in reader:
        assert len(row['ticker'])>2 and row['permno'] in stats['overlaps_2012_2022']['permnos']
        assert overlaps(date.fromisoformat(row['secinfostartdt']),date.fromisoformat(row['secinfoenddt']))
        output_permnos.add(row['permno'])
        checked+=1
        if len(preview)<5:preview.append(row)
assert checked==stats['overlaps_2012_2022']['rows']
assert output_permnos==set(read_permnos)
assert input_stat==(SOURCE.stat().st_size,SOURCE.stat().st_mtime_ns)
summary={'status':'complete','source':str(SOURCE),'source_size_mtime':input_stat,
         'research_start':START.isoformat(),'research_end':END.isoformat(),'endpoints_inclusive':True,
         'permno_scope':'records after ticker length >2 and date overlap filters','ticker_history_deduplicated':False,
         'ticker_file_delimiter':'tab','original_dates_preserved':True,
         'diagnostics':[diagnostic(stage) for stage in stats],
         'excluded_short_or_empty_ticker_rows':stats['input']['rows']-stats['ticker_length_gt_2']['rows'],
         'excluded_nonoverlapping_rows_after_length_filter':stats['ticker_length_gt_2']['rows']-checked,
         'permno_file':permno_path.name,'ticker_file':ticker_path.name}
(OUT/'run_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
display(pd.DataFrame(summary['diagnostics']))
display(pd.DataFrame(preview))
print('驗證通過：',len(permnos),'個唯一 PERMNO；',checked,'筆 ticker 歷史紀錄。')
print('輸出：',OUT)


,stage,rows,start_min,end_max,unique_permnos,unique_tickers,unique_authors,unique_threads
0,input,110365,1925-12-31,2025-12-31,26907,26216,N/A,N/A
1,ticker_length_gt_2,104005,1945-12-31,2025-12-31,25337,25694,N/A,N/A
2,overlaps_2012_2022,16508,2008-06-27,2023-09-28,6675,7314,N/A,N/A


,ticker,secinfostartdt,secinfoenddt,permno
0,EGAS,2010-07-09,2017-08-03,10001
1,BTFG,2008-06-27,2013-02-15,10002
2,AEPI,2009-03-27,2017-01-20,10025
3,JJSF,2008-06-27,2019-08-04,10026
4,JJSF,2019-08-05,2021-04-26,10026


驗證通過： 6675 個唯一 PERMNO； 16508 筆 ticker 歷史紀錄。
輸出： C:\Users\eagle\RedditStock\Data\process\00_biild_ticker_list\2026-10-05_08-12-26_UTC
